In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[1] # go up n levels (adjust as needed)
sys.path.append(str(ROOT))

from config import PROJECT_ROOT, APT_ROOT
from apt_project import *

In [2]:
import json
import pandas as pd
from pprint import pprint
import matplotlib.pyplot as plt
import networkx as nx
import seaborn as sns
import numpy as np

Actually doing technique diversity

In [3]:
# Computes normalized shannnon entropy of techniques used by a group
def normalized_entropy(df):
    counts = df['count'].values
    p = counts / counts.sum()
    # Shannon entropy
    H = -(p * np.log(p)).sum()
    # Normalize by log(K)
    K = df['kill_chain_phases'].nunique()
    return H / np.log(K)

In [4]:
merged = group_techniques_df.merge(
    tech_df,
    left_on='technique_id',
    right_on='id',
    how='left'
)

# Expand kill chain phase lists into separate rows
merged = merged.explode('kill_chain_phases')


phase_counts = (
    merged.groupby(['group_id', 'group_name', 'kill_chain_phases'])
          .size()
          .reset_index(name='count')
)


def normalized_entropy(df):
    counts = df['count'].values
    p = counts / counts.sum()
    H = -(p * np.log(p)).sum()
    K = df['kill_chain_phases'].nunique()
    return H / np.log(K) if K > 1 else 0.0  # if only 1 phase, entropy = 0


entropy_df = (
    phase_counts.groupby(['group_id', 'group_name'])
                .apply(normalized_entropy)
                .reset_index(name='normalized_entropy')
)


TOTAL_PHASES = 14

coverage_df = (
    phase_counts.groupby(['group_id', 'group_name'])['kill_chain_phases']
                .nunique()
                .reset_index(name='num_phases')
)

coverage_df['coverage'] = coverage_df['num_phases'] / TOTAL_PHASES

final_df = entropy_df.merge(
    coverage_df[['group_id', 'coverage']],
    on='group_id',
    how='left'
)

# Weighted scoring: Coverage = 0.8, Entropy = 0.2
final_df['final_score'] = (
    0.8 * final_df['coverage'] +
    0.2 * final_df['normalized_entropy']
)


# Min-max scale the final_score to 0–1
final_df['scaled_score'] = (
    (final_df['final_score'] - final_df['final_score'].min()) /
    (final_df['final_score'].max() - final_df['final_score'].min())
)

# Optional: sort by scaled_score (highest first)
final_df = final_df.sort_values('scaled_score', ascending=False).reset_index(drop=True)

final_df

/var/folders/76/8sr7whp91112ln7fv5bsf9kc0000gn/T/ipykernel_38202/1298197509.py:29: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(normalized_entropy)


,group_id,group_name,normalized_entropy,coverage,final_score,scaled_score
0,intrusion-set--a7f57cc1-4540-4429-823f-f4e56b8...,Ember Bear,0.960150,1.000000,0.992030,1.000000
1,intrusion-set--381fcf73-60f6-4ab2-9991-6af3cbc...,Sandworm Team,0.953578,1.000000,0.990716,0.998594
2,intrusion-set--44d37b89-a739-4810-9111-0d2617a...,Scattered Spider,0.944924,1.000000,0.988985,0.996743
3,intrusion-set--bef4c620-0787-42a8-a96d-b7eb6e8...,APT28,0.939676,1.000000,0.987935,0.995620
4,intrusion-set--01e28736-2ffc-455b-9880-ed4d140...,Indrik Spider,0.930670,1.000000,0.986134,0.993693
...,...,...,...,...,...,...
163,intrusion-set--0ea72cd5-ca30-46ba-bc04-378f701...,GCMAN,0.000000,0.071429,0.057143,0.000000
164,intrusion-set--090242d7-73fc-4738-af68-20162f7...,APT17,0.000000,0.071429,0.057143,0.000000
165,intrusion-set--d6e88e18-81e8-4709-82d8-973095d...,APT16,0.000000,0.071429,0.057143,0.000000
166,intrusion-set--2e5d3a83-fe00-41a5-9b60-237efc8...,Moafee,0.000000,0.071429,0.057143,0.000000


CSV exporting

In [ ]:
# csv_df = final_df[['group_name', 'scaled_score']]
# csv_df.rename(columns={'scaled_score': 'score'}, inplace=True)

/var/folders/76/8sr7whp91112ln7fv5bsf9kc0000gn/T/ipykernel_38202/3086918356.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  csv_df.rename(columns={'scaled_score': 'score'}, inplace=True)


,group_name,score
0,Ember Bear,1.000000
1,Sandworm Team,0.998594
2,Scattered Spider,0.996743
3,APT28,0.995620
4,Indrik Spider,0.993693
...,...,...
163,GCMAN,0.000000
164,APT17,0.000000
165,APT16,0.000000
166,Moafee,0.000000


In [ ]:
# csv_df.to_csv("../analysis_data/technique_diversity.csv", index=False)